# Pancancer enrichment analysis step 5: Find enriched pathways using GSEApy

## Setup

In [1]:
import cptac
import cptac.utils as ut
import pandas as pd
import numpy as np
import gseapy as gp
import os
import datetime
import glob

In [2]:
TIME_START = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')

STEP4_DIR = "step4_outputs"

STEP5_DIR = "step5_outputs"
STEP5_FILE = f"enrichment_gseapy_{TIME_START}.tsv"
STEP5_FILE_PATH = os.path.join(STEP5_DIR, STEP5_FILE)

GSEAPY_DIR_PATH = os.path.join(STEP5_DIR, "gseapy")

if not os.path.isdir(STEP5_DIR):
    os.mkdir(STEP5_DIR)

## Prepare data

In [3]:
# Read in the files from step 4
step4_files = glob.glob(f"{STEP4_DIR}{os.sep}*")
prot_dict = {}
for file in step4_files:
    name = file.split(os.sep)[1].split("_")[0]
    prot_dict[name] = pd.read_csv(file, sep='\t')

In [4]:
prot_dict.keys()

dict_keys(['hnscc', 'lscc', 'colon', 'endometrial', 'luad', 'ccrcc', 'gbm', 'ovarian'])

## Run enrichment analysis

In [5]:
# For each cancer, find enriched pathways.
all_enrichments = pd.DataFrame()

for cancer_type in prot_dict.keys():
    
    prot = prot_dict[cancer_type]
    samples = prot.columns[~prot.columns.isin(["NAME"])]
    cls_list = np.where(samples.str.endswith(".N"), "Normal", "Tumor").tolist()

    gs_res = gp.gsea(
        data=prot,
        gene_sets="gene_set_libraries/KEGG_2019_Human.gmt",
        cls=cls_list,
        permutation_type="phenotype",
        permutation_num=10,
        min_size=1,
        max_size=1000, 
        outdir=GSEAPY_DIR_PATH,
        no_plot=True,
        method="abs_signal_to_noise",
        processes=1,
        seed=0)
    
    cancer_enriched = gs_res.res2d.assign(cancer_type=cancer_type)
    all_enrichments = all_enrichments.append(cancer_enriched)

2020-06-16 11:59:28,171 Warning: Input data contains NA, filled NA with 0
2020-06-16 12:01:04,589 Warning: Input data contains NA, filled NA with 0
2020-06-16 12:02:38,272 Warning: Input data contains NA, filled NA with 0
2020-06-16 12:03:40,172 Warning: Input data contains NA, filled NA with 0
2020-06-16 12:05:05,088 Warning: Input data contains NA, filled NA with 0
2020-06-16 12:06:31,853 Warning: Input data contains NA, filled NA with 0
2020-06-16 12:08:00,352 Warning: Input data contains NA, filled NA with 0
2020-06-16 12:09:23,067 Warning: Input data contains NA, filled NA with 0


In [6]:
all_enrichments.to_csv(STEP5_FILE_PATH, sep="\t")